# Mixing-score correlation with feedforward controls

How closely does each recurrence of a looped model reproduce the mixing-score
profile of its matched feedforward control? For each main-paper model, take the
layer profile formed by the prelude, **one recurrence**, and the coda, and
correlate it with the control's profile. Each panel has one bar per recurrence
and a final bar for the mean over the displayed recurrences. One figure uses
Pearson's $r$ and a second uses Spearman's $\rho$.

`mode="quick"` defaults to one seeded example and saves figures only in the run
directory. `mode="full"` defaults to 256 examples and also updates the paper
figures. Both modes use the same model–recurrence pairs and figures; an explicit
`limit` overrides the sample count, independently of export policy.

In [ ]:
mode = "quick"  # "quick" saves locally; "full" also updates paper figures.
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
# The runs behind the paper's within-recurrence mixing-score summary.
model_recurrences = [
    ("ouro_1_4b", 4),
    ("raven", 32),
    ("retrofitted_llama", 32),
    ("parcae_770m", 32),
]
shown_recurrences = 8  # Bars per panel, from the first recurrence; Ouro has four.
limit = None  # Default: 1 (quick), 256 (full); 0 means the entire split.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict, replace
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr

from experiments.cache import ExperimentCache
from experiments.support import (
    BASELINE_FOR,
    MODEL_LABELS,
    collect_metrics,
    configure_experiment,
    export_paper_figures,
    model_subplots,
    save_figure,
    save_results,
    summarize_stages,
)

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

model_recurrences = [tuple(pair) for pair in model_recurrences]
control_pairs = list(
    dict.fromkeys((BASELINE_FOR[name], 1) for name, _ in model_recurrences)
)
sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
# Collect exactly as stages of inference does (including its residual entropy),
# so completed stages runs are reused from the cache without inference.
configs = {
    (name, recurrences): configure_experiment(
        experiment="stages_of_inference",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in [*model_recurrences, *control_pairs]
}
config = replace(next(iter(configs.values())), experiment="mixing_correlation")

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
STATISTICS = {
    "pearson": ("Pearson's $r$", pearsonr),
    "spearman": ("Spearman's $\\rho$", spearmanr),
}


def recurrence_profiles(example, recurrences):
    """Mixing scores of prelude + one recurrence + coda, for the first recurrences."""
    layers = np.asarray(example["layers"])
    scores = np.asarray(example["curves"]["mixing_scores"])
    core = example["recurrent_layers"]
    prelude = np.flatnonzero(layers < min(core))
    coda = np.flatnonzero(layers > max(core))
    blocks = np.flatnonzero(np.isin(layers, core)).reshape(-1, len(core))
    return np.stack(
        [scores[np.r_[prelude, block, coda]] for block in blocks[:recurrences]]
    )


def control_at_depths(control, n_layers):
    """Linearly interpolate control profiles at the looped model's layer depths."""
    depths = np.linspace(0, 1, n_layers)
    return np.stack([np.interp(depths, np.linspace(0, 1, len(c)), c) for c in control])


def correlations(looped, control, statistic):
    """Correlate every recurrence with the control; append the recurrence mean.

    ``looped`` has shape (examples, recurrences, layers) and ``control`` has shape
    (examples, layers), already sampled at the same depths.
    """
    correlate = STATISTICS[statistic][1]
    of_means = np.array(
        [correlate(r, control.mean(0)).statistic for r in looped.mean(0)]
    )
    per_example = np.array(
        [
            [correlate(r, c).statistic for r in profiles]
            for profiles, c in zip(looped, control, strict=True)
        ]
    )
    return {
        "of_means": np.append(of_means, of_means.mean()),
        "per_example": np.column_stack([per_example, per_example.mean(axis=1)]),
    }

## Generate and reduce

Each looped model and each matched control runs once on the same seeded examples:
Llama 3.2 1B for Ouro, Huginn, and Retrofitted Llama, and the trained one-pass
Parcae 770M for Parcae. Collection is identical to stages of inference, so full
runs reuse its cached per-example curves; residual entropy is computed only
because that shared collection already includes it.

Retain each example's mixing-score profiles, compute both correlation summaries,
and save them with run metadata in `overview.pt` and `correlations.csv`. Figure
cells reuse these values without inference.

In [ ]:
settings = asdict(config)
del settings["models"], settings["num_steps"]
settings.update(
    model_recurrences=model_recurrences,
    control_pairs=control_pairs,
    shown_recurrences=shown_recurrences,
    output_dir=str(config.output_dir),
    versions={
        name: version(name) for name in ("torch", "transformer-lens", "transformers")
    },
    cache=result_cache.settings,
)
run_dir = save_results({}, settings, config.output_dir)

profiles, run_info = {}, {}
for pair, pair_config in configs.items():
    results, run_info[pair] = collect_metrics(
        pair_config, summarize_stages, cache=result_cache
    )
    examples = results[pair[0]]
    if pair in control_pairs:
        profiles[pair] = np.array([e["curves"]["mixing_scores"] for e in examples])
    else:
        profiles[pair] = np.array(
            [recurrence_profiles(e, shown_recurrences) for e in examples]
        )
    del results, examples

correlation = {statistic: {} for statistic in STATISTICS}
for pair in model_recurrences:
    looped = profiles[pair]
    control = control_at_depths(profiles[(BASELINE_FOR[pair[0]], 1)], looped.shape[-1])
    for statistic in STATISTICS:
        correlation[statistic][pair] = correlations(looped, control, statistic)

torch.save(
    {"profiles": profiles, "correlation": correlation, "run": run_info},
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the bars measure

Each mixing score is the attention-row entropy divided by its causal maximum,
averaged over query rows excluding the first, then over heads within an example.
For recurrence $k$, the looped profile lists the prelude, the $k$-th application
of the recurrent block, and the coda, in layer order. Ouro has no prelude or coda,
so its profile is a single recurrence. Each recurrence is unaffected by the
rollout length; the coda follows the final recurrence of the 32-step run.

Layer depth places the first and last layers at 0% and 100%. Where the looped and
control models have different layer counts, linearly interpolate the control at
the looped model's layer depths, then correlate across those depths:

| Model | Looped layers per profile | Control layers |
| --- | --- | --- |
| Ouro 1.4B | 24 | 16 (Llama 3.2 1B) |
| Huginn-0125 | 2 + 4 + 2 = 8 | 16 (Llama 3.2 1B) |
| Retrofitted Llama | 4 + 6 + 4 = 14 | 16 (Llama 3.2 1B) |
| Parcae 770M | 6 + 6 + 6 = 18 | 18 (Parcae 770M feedforward) |

Each panel shows two summaries:

| Bar | Definition | Question answered |
| --- | --- | --- |
| Mean profiles | Average the profiles over examples, then correlate the two means. | Do the averaged curves in the paper's profile figure have the same shape? |
| Per example | Correlate the two profiles of each example, then average over examples. Whiskers show one sample standard deviation. | How reliably does a single input reproduce that shape? |

Pearson's $r$ measures linear agreement in the profile's shape and is insensitive
to offsets and scale. Spearman's $\rho$ compares only the rank order of layers, so
it ignores monotone distortions. The final bar is the arithmetic mean over the
displayed recurrences: of the mean-profile correlations, or of each example's
correlations before averaging examples. Correlations use only 8–24 depths, so
small layer-count differences between profiles matter.

In [ ]:
def correlation_table(pair):
    columns = {}
    for statistic, values in correlation.items():
        columns[(statistic, "mean profiles")] = values[pair]["of_means"]
        columns[(statistic, "per example")] = values[pair]["per_example"].mean(axis=0)
    n = len(values[pair]["of_means"]) - 1
    return pd.DataFrame(columns, index=[*map(str, range(1, n + 1)), "Mean"])


table = pd.concat(
    {MODEL_LABELS[pair[0]]: correlation_table(pair) for pair in model_recurrences},
    names=["Model", "Recurrence"],
)
table.to_csv(run_dir / "correlations.csv")
table.round(3)

In [ ]:
def plot_correlations(statistic):
    """One panel per model: bars per recurrence, then their mean."""
    label = STATISTICS[statistic][0]
    fig, axes = model_subplots(len(model_recurrences), height_per_subplot=1.5)
    width = 0.4
    for ax, pair in zip(axes, model_recurrences, strict=True):
        values = correlation[statistic][pair]
        per_example = values["per_example"]
        n = per_example.shape[1] - 1
        x = np.append(np.arange(1, n + 1), n + 1.6)  # Gap before the mean.
        ax.bar(
            x - width / 2, values["of_means"], width, color="C0", label="Mean profiles"
        )
        ax.bar(
            x + width / 2,
            per_example.mean(axis=0),
            width,
            yerr=per_example.std(axis=0, ddof=1 if len(per_example) > 1 else 0),
            color="C1",
            error_kw=dict(elinewidth=0.6, capsize=1),
            label="Per example (mean ± SD)",
        )
        ax.axhline(0, color="black", linewidth=0.6)
        ax.axvline(n + 0.8, color="grey", linestyle="--", linewidth=0.6)
        ax.set(
            title=MODEL_LABELS[pair[0]],
            xlabel="Recurrence",
            xticks=x,
            xticklabels=[*map(str, range(1, n + 1)), "Mean"],
            ylim=(-1, 1),
        )
        ax.tick_params(axis="x", which="both", top=False)
        ax.minorticks_off()
        ax.grid(axis="x", visible=False)
        if ax.get_subplotspec().is_first_col():
            ax.set_ylabel(label)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncols=2)
    save_figure(fig, run_dir, f"mixing_score_{statistic}", bbox_inches=None)

## Pearson correlation

In [ ]:
plot_correlations("pearson")

## Spearman correlation

In [ ]:
plot_correlations("spearman")

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")